# GitHub RAG Evaluator — Google Colab

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

GPU cuts embedding time from ~7 minutes → ~60 seconds.

In [ ]:
# ── Step 1: Install dependencies ─────────────────────────────────────────────
!pip install -q sentence-transformers==2.7.0 faiss-cpu==1.8.0 google-generativeai==0.7.2 numpy==1.26.4

In [ ]:
# ── Step 2: Clone YOUR project from GitHub ───────────────────────────────────
# Replace with your actual GitHub repo URL
!git clone https://github.com/YOUR_USERNAME/github-rag-evaluator.git
%cd github-rag-evaluator

In [ ]:
# ── Step 3: Set your Gemini API key ──────────────────────────────────────────
# Option A — Colab Secrets (recommended, key stays private)
# Click the 🔑 key icon on the left sidebar → Add secret → Name: GEMINI_API_KEY
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("API key set ✅")

# Option B — paste directly (less safe, don't share the notebook after)
# os.environ["GEMINI_API_KEY"] = "AIzaSy_paste_your_key_here"

In [ ]:
# ── Step 4: Fix the embedder for this environment ────────────────────────────
import sys
sys.path.insert(0, "src")
sys.path.insert(0, "eval")

# Patch embedder to use GPU if available
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print("✅ GPU detected — embedding will be ~7x faster")
else:
    print("⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Step 5: Run a single query ───────────────────────────────────────────────
!python main.py --query "How do I declare path parameters?"

In [ ]:
# ── Step 6: Try other queries ────────────────────────────────────────────────
!python main.py --query "How does dependency injection work?"
# !python main.py --query "How do I handle file uploads?" --strategy semantic --reranker
# !python main.py --query "How does OAuth2 work in FastAPI?" --strategy fixed_with_overlap

In [ ]:
# ── Step 7: Run FULL evaluation (all 6 strategy combinations) ────────────────
# This produces the numbers for your README table
# Takes ~5-10 min on GPU, ~30 min on CPU
!python main.py --evaluate

In [ ]:
# ── Step 8: Download the results JSON ────────────────────────────────────────
from google.colab import files
files.download("results/evaluation_results.json")

In [ ]:
# ── Step 9: Print the README table ───────────────────────────────────────────
import json
with open("results/evaluation_results.json") as f:
    results = json.load(f)

print("\nCopy this into your README:\n")
print(f"| {'Strategy':<25} | {'Reranker':<10} | {'Hit Rate'} |")
print(f"| {'-'*25} | {'-'*10} | {'-'*8} |")
for r in results:
    print(f"| {r['strategy']:<25} | {str(r['use_reranker']):<10} | {r['hit_rate']}%     |")